# Week 4: Conversational RAG — LangChain LCEL + Memory + Structured Output

A fully self-contained walk-through of the Week 4 pipeline.

| Component | Choice |
|---|---|
| **Retriever** | Local ChromaDB (all-MiniLM-L6-v2) via `langchain-chroma` |
| **Primary LLM** | Gemini 2.5 Flash via `ChatGoogleGenerativeAI` |
| **Structured Output** | Pydantic `FinancialSnapshot` bound with `.with_structured_output()` |
| **Memory** | `ConversationSummaryBufferMemory` (2,000 token limit, Groq Llama-3 summariser) |
| **Chain** | LangChain 1.x LCEL `RunnablePassthrough` pipeline |


## Step 0 — Imports & Environment Setup

In [1]:
import os, sys, json, pathlib
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.memory import ConversationSummaryBufferMemory

from src.tools.vector_store import get_langchain_retriever

print("All imports OK.")

c:\Users\multi\genai_sos\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK.


## Step 1 — Define the Structured Output Schema (`FinancialSnapshot`)

We define a strict Pydantic class whose fields map exactly to what Gemini **must** return.
Binding it via `.with_structured_output()` forces the model to emit a valid JSON object — no loose prose.

In [2]:
class FinancialSnapshot(BaseModel):
    """Strict structured output for a company financial profile."""
    company_name:     str = Field(description='Name of the company being analysed.')
    total_revenue:    str = Field(description='Total revenue / net-sales figures from the retrieved context.')
    debt_obligations: str = Field(description='Long-term or short-term debt, lease, or purchase obligations.')
    noted_risks:      str = Field(description='Key business, legal, or market risks from the retrieved context.')

print(FinancialSnapshot.model_json_schema())

{'description': 'Strict structured output for a company financial profile.', 'properties': {'company_name': {'description': 'Name of the company being analysed.', 'title': 'Company Name', 'type': 'string'}, 'total_revenue': {'description': 'Total revenue / net-sales figures from the retrieved context.', 'title': 'Total Revenue', 'type': 'string'}, 'debt_obligations': {'description': 'Long-term or short-term debt, lease, or purchase obligations.', 'title': 'Debt Obligations', 'type': 'string'}, 'noted_risks': {'description': 'Key business, legal, or market risks from the retrieved context.', 'title': 'Noted Risks', 'type': 'string'}}, 'required': ['company_name', 'total_revenue', 'debt_obligations', 'noted_risks'], 'title': 'FinancialSnapshot', 'type': 'object'}


## Step 2 — Initialise LLMs

Two LLMs serve distinct roles:
- **Gemini 2.5 Flash** — primary generator with structured output
- **Groq Llama-3** — background memory summariser (compresses old turns when the 2,000-token buffer fills)

In [3]:
# Primary: Gemini 2.5 Flash with FinancialSnapshot schema bound
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=os.getenv("GEMINI_API_KEY")
)
structured_llm = gemini_llm.with_structured_output(FinancialSnapshot)

# Background summariser: Groq Llama-3 (fast, cheap, keeps memory compressed)
groq_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Gemini primary LLM  : ready")
print("Groq summariser     : ready")

Gemini primary LLM  : ready
Groq summariser     : ready


## Step 3 — Connect the ChromaDB Retriever

`get_langchain_retriever()` wraps our pre-ingested ChromaDB collection (`ma_due_diligence`) with `langchain-chroma` and the same `all-MiniLM-L6-v2` embedding model used during Week 3 ingestion — so no re-embedding is needed.

We use `top_k=30` to ensure enough diverse chunks are retrieved to cover the financial figures.

In [4]:
retriever = get_langchain_retriever(persist_dir='../outputs/chromadb', top_k=30)

# Sanity check
sample_docs = retriever.invoke("Apple revenue")
print(f"Retrieved {len(sample_docs)} docs for sanity check:")
for d in sample_docs[:3]:
    company  = d.metadata.get('company', '?')
    section  = d.metadata.get('section', '?')
    snippet  = d.page_content[:80]
    print(f"  [{company} | {section}] {snippet}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7891.66it/s]


Retrieved 30 docs for sanity check:
  [Apple | Income_Tax] Apple Inc. | 2025 Form 10-K | 40
A reconciliation of the provision for income ta...
  [Apple | General] (1)
Apple Inc. | 2025 Form 10-K | 23
Gross Margin
Products and Services gross ma...
  [Apple | General] Apple Inc.
CONSOLIDATED STATEMENTS OF OPERATIONS
(In millions, except number of ...


## Step 4 — Initialise `ConversationSummaryBufferMemory`

`ConversationSummaryBufferMemory` keeps the last N tokens of chat history verbatim.
Once the buffer exceeds `max_token_limit=2000`, Groq Llama-3 automatically compresses
older turns into a rolling summary paragraph — keeping total context size bounded.

`return_messages=True` ensures history is returned as `BaseMessage` objects,
compatible with the `MessagesPlaceholder` in our chat prompt.

In [5]:
memory = ConversationSummaryBufferMemory(
    llm=groq_llm,
    max_token_limit=2000,
    return_messages=True,
    memory_key="chat_history"
)

print("Memory initialised.")
print(memory.load_memory_variables({}))

Memory initialised.
{'chat_history': []}


C:\Users\multi\AppData\Local\Temp\ipykernel_34712\368043383.py:1: LangChainDeprecationWarning: The class `ConversationSummaryBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryBufferMemory(


## Step 5 — Build the LCEL Chain

The chain is built with LangChain 1.x **LCEL** (LangChain Expression Language):

```
RunnablePassthrough.assign(context = retrieve_and_format)
  | ChatPromptTemplate
  | structured_llm  (Gemini + FinancialSnapshot)
```

This is the modern equivalent of the legacy `create_retrieval_chain` from LangChain 0.3.x.

In [6]:
def _format_docs(docs) -> str:
    """Concatenate retrieved docs with source metadata headers."""
    parts = []
    for d in docs:
        m = d.metadata
        company = m.get('company', '??')
        section = m.get('section', '??')
        header  = f'[{company} | {section}]'
        parts.append(f'{header}\n{d.page_content}')
    return '\n\n---\n\n'.join(parts)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "You are an expert M&A financial analyst. "
            "Answer using ONLY the following retrieved context. "
            "If data is not present in the context, output 'N/A'. "
            "Do NOT invent numbers.\n\nContext:\n{context}"
        ),
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: _format_docs(retriever.invoke(x["input"]))
    )
    | prompt
    | structured_llm
)

print("LCEL chain built successfully.")

LCEL chain built successfully.


## Step 6 — Run Multi-Turn Conversational Queries

Each turn:
1. Loads the accumulated `chat_history` from memory
2. Invokes the LCEL chain — retrieves context, fills the prompt, calls Gemini
3. Receives a `FinancialSnapshot` Pydantic object (strict JSON)
4. Saves the exchange back into memory (Groq compresses if over 2,000 tokens)

In [7]:
queries = [
    "What is Apple's total revenue and what are their most significant noted risks?",
    "Now give me the same breakdown for Nvidia. Pay attention to their debt obligations too.",
    "Compare the risk profiles of Apple and Nvidia based on what we just discussed.",
]

results = []

for i, q in enumerate(queries, 1):
    print(f"--- Turn {i} ---")
    print(f"User: {q}\n")

    history = memory.load_memory_variables({})

    answer: FinancialSnapshot = rag_chain.invoke({
        "input": q,
        "chat_history": history.get("chat_history", [])
    })

    answer_dict = answer.model_dump()
    memory.save_context({"input": q}, {"output": json.dumps(answer_dict)})
    results.append({"turn": i, "query": q, "structured_answer": answer_dict})

    print("System (structured JSON):")
    print(json.dumps(answer_dict, indent=2))
    print()

--- Turn 1 ---
User: What is Apple's total revenue and what are their most significant noted risks?



c:\Users\multi\genai_sos\.venv\Lib\site-packages\langchain_core\language_models\base.py:447: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


System (structured JSON):
{
  "company_name": "Apple Inc.",
  "total_revenue": "$416,161 million for 2025",
  "debt_obligations": "Term debt includes fixed-rate notes totaling $86,781 million as of September 27, 2025, with maturities from 2025 to 2062.",
  "noted_risks": "Key risks include the inability to compete successfully, challenges in managing frequent product and service introductions, significant risks of supply shortages and price increases, exposure to various legal proceedings and claims, potential impacts from political events, trade disputes, and geopolitical tensions, design and manufacturing defects in products and services, intense media, political and regulatory scrutiny, cybersecurity vulnerabilities (hacking, ransomware attacks, employee error), and dependence on third-party software developers and resellers."
}

--- Turn 2 ---
User: Now give me the same breakdown for Nvidia. Pay attention to their debt obligations too.

System (structured JSON):
{
  "company_name":

## Step 7 — Inspect the Memory Buffer

After 3 turns the buffer may still be under 2,000 tokens, so history appears verbatim.
After more turns Groq would compress it into a rolling summary paragraph.

In [8]:
final_memory = memory.load_memory_variables({})
print("Current memory state:")
for msg in final_memory.get('chat_history', []):
    role = msg.__class__.__name__
    preview = str(msg.content)[:120]
    print(f"  [{role}]: {preview}...")

Current memory state:
  [HumanMessage]: What is Apple's total revenue and what are their most significant noted risks?...
  [AIMessage]: {"company_name": "Apple Inc.", "total_revenue": "$416,161 million for 2025", "debt_obligations": "Term debt includes fix...
  [HumanMessage]: Now give me the same breakdown for Nvidia. Pay attention to their debt obligations too....
  [AIMessage]: {"company_name": "NVIDIA Corporation", "total_revenue": "$215,938 million for Jan 25, 2026", "debt_obligations": "Debt i...
  [HumanMessage]: Compare the risk profiles of Apple and Nvidia based on what we just discussed....
  [AIMessage]: {"company_name": "Apple Inc. vs. NVIDIA Corporation", "total_revenue": "N/A", "debt_obligations": "N/A", "noted_risks": ...


## Step 8 — Save Outputs

In [9]:
out_dir = pathlib.Path('../outputs')
(out_dir / 'responses').mkdir(parents=True, exist_ok=True)
(out_dir / 'reports').mkdir(parents=True, exist_ok=True)

# JSON responses
json_path = out_dir / 'responses' / 'week4_responses.json'
json_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
print(f'Saved JSON responses -> {json_path}')

# Markdown report
report_lines = [
    '# Week 4: Conversational RAG with Memory & Structured Outputs\n',
    '## Architecture\n',
    '| Component | Choice |\n',
    '|---|---|\n',
    '| **Retriever** | Local ChromaDB (all-MiniLM-L6-v2) via `langchain-chroma` |\n',
    '| **Primary LLM** | Gemini 2.5 Flash via `langchain-google-genai` |\n',
    '| **Structured Output** | Pydantic `FinancialSnapshot` via `.with_structured_output()` |\n',
    '| **Memory** | `ConversationSummaryBufferMemory` (2,000 tokens, Groq summariser) |\n',
    '| **Chain** | LCEL `RunnablePassthrough` (LangChain 1.x) |\n\n',
    '## Conversation Logs\n\n',
]
for r in results:
    report_lines.append(f"### Turn {r['turn']}\n")
    report_lines.append(f"**User:** {r['query']}\n\n")
    report_lines.append('**System:**\n```json\n')
    report_lines.append(json.dumps(r['structured_answer'], indent=2))
    report_lines.append('\n```\n\n')

report_path = out_dir / 'reports' / 'week4_report.md'
report_path.write_text(''.join(report_lines), encoding='utf-8')
print(f'Saved report         -> {report_path}')

Saved JSON responses -> ..\outputs\responses\week4_responses.json
Saved report         -> ..\outputs\reports\week4_report.md
